# 📊 ML Model Monitoring & Drift Detection

A practical portfolio notebook that trains a baseline classifier, simulates production batches, detects feature drift, checks data quality, monitors predictions and performance, and exports an alert report. Uses synthetic data; no API key or external dataset required.

## Learning goals
- Establish reference data and baseline metrics.
- Detect numeric drift with the Kolmogorov–Smirnov (KS) test.
- Detect categorical mix changes with a chi-square test.
- Monitor missing values, prediction rates, and performance when labels are available.
- Generate configurable alerts and export CSV/JSON results.

> Note: Data drift is not proof of model failure. True performance evaluation requires reliable ground-truth labels.

In [ ]:
%pip -q install numpy pandas scikit-learn matplotlib scipy

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, chisquare
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
rng = np.random.default_rng(SEED)
print('Imports complete')

## 1. Create reference data and train a baseline model

In [ ]:
X, y = make_classification(n_samples=6000, n_features=8, n_informative=5, n_redundant=1, weights=[0.62, 0.38], flip_y=0.02, random_state=SEED)
features = [f'feature_{i+1}' for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=features)
df['region'] = rng.choice(['north','south','east','west'], len(df), p=[.25,.30,.20,.25])
df['target'] = y
train, holdout = train_test_split(df, test_size=.25, random_state=SEED, stratify=df.target)
model = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('classifier', LogisticRegression(max_iter=1000, random_state=SEED))])
model.fit(train[features], train.target)
base_pred = model.predict(holdout[features]); base_prob = model.predict_proba(holdout[features])[:,1]
baseline = {'accuracy':accuracy_score(holdout.target,base_pred),'precision':precision_score(holdout.target,base_pred,zero_division=0),'recall':recall_score(holdout.target,base_pred,zero_division=0),'f1':f1_score(holdout.target,base_pred,zero_division=0),'roc_auc':roc_auc_score(holdout.target,base_prob)}
display(pd.Series(baseline, name='Holdout baseline').to_frame())
display(df.head())

## 2. Simulate production batches

The stable batch is sampled from the same distribution. Other batches introduce controlled numeric and categorical shifts; the strongest batch also changes labels and adds missing values to demonstrate separate monitoring signals.

In [ ]:
def make_batch(seed, strength=0.0, label_shift=False, n=900):
    batch = train.sample(n=n, replace=True, random_state=seed).copy().reset_index(drop=True)
    local_rng = np.random.default_rng(seed)
    batch['feature_1'] += strength * 1.4
    batch['feature_2'] *= (1 + strength * .65)
    batch['feature_4'] += strength * .9
    if strength > 0:
        batch['region'] = local_rng.choice(['north','south','east','west'], n, p=[.55,.15,.10,.20])
    if label_shift:
        batch['target'] = 1 - batch['target']
    if strength >= 1:
        ids = local_rng.choice(n, size=max(1, int(n*.025)), replace=False)
        batch.loc[ids, 'feature_3'] = np.nan
    return batch

batches = {'stable_batch':make_batch(101,0), 'moderate_drift':make_batch(202,.8), 'strong_drift':make_batch(303,1.7,True)}
{key: value.shape for key,value in batches.items()}

## 3. Data quality report

In [ ]:
def quality_report(batch):
    return {'rows':len(batch), 'duplicate_rows':int(batch.duplicated().sum()), 'missing_columns':sorted(set(train.columns)-set(batch.columns)), 'unexpected_columns':sorted(set(batch.columns)-set(train.columns)), 'missing_percent':(batch[features].isna().mean()*100).round(2).to_dict()}
for name,batch in batches.items():
    print('\n',name, {k:v for k,v in quality_report(batch).items() if k!='missing_percent'})
    display(pd.Series(quality_report(batch)['missing_percent'], name='missing_%').to_frame())

## 4. Numeric drift with KS test

A Bonferroni-adjusted significance level is used across numeric features. Pair p-values with effect sizes and domain thresholds in real deployments.

In [ ]:
def numeric_drift(reference, current, columns, alpha=.05):
    rows=[]; adjusted_alpha=alpha/max(1,len(columns))
    for col in columns:
        a=reference[col].dropna().to_numpy(); b=current[col].dropna().to_numpy()
        stat,p=ks_2samp(a,b)
        rows.append({'feature':col,'ks_statistic':stat,'p_value':p,'drift_detected':p<adjusted_alpha,'adjusted_alpha':adjusted_alpha})
    return pd.DataFrame(rows).sort_values('p_value')

numeric_reports={}
for name,batch in batches.items():
    numeric_reports[name]=numeric_drift(train[features],batch[features],features)
    print(name); display(numeric_reports[name].round(4))

## 5. Categorical drift with chi-square test

In [ ]:
def categorical_drift(reference, current, column, alpha=.05):
    cats=sorted(set(reference[column].dropna().unique())|set(current[column].dropna().unique()))
    ref=reference[column].value_counts().reindex(cats,fill_value=0).astype(float)
    cur=current[column].value_counts().reindex(cats,fill_value=0).astype(float)
    expected=ref/ref.sum()*cur.sum()+1e-8
    stat,p=chisquare(cur.to_numpy(), expected.to_numpy())
    return {'feature':column,'chi_square':float(stat),'p_value':float(p),'drift_detected':bool(p<alpha),'reference_mix':(ref/ref.sum()).round(3).to_dict(),'current_mix':(cur/cur.sum()).round(3).to_dict()}

cat_reports={}
for name,batch in batches.items():
    cat_reports[name]=categorical_drift(train,batch,'region')
    print(name, cat_reports[name])

## 6. Prediction drift and model performance

Prediction statistics can be computed before labels arrive. Accuracy and other performance metrics require ground-truth labels.

In [ ]:
def evaluate_batch(batch):
    pred=model.predict(batch[features]); prob=model.predict_proba(batch[features])[:,1]
    result={'rows':len(batch),'positive_prediction_rate':float(np.mean(pred==1)),'mean_positive_probability':float(prob.mean())}
    if batch.target.notna().all():
        y_true=batch.target.astype(int)
        result.update({'accuracy':accuracy_score(y_true,pred),'precision':precision_score(y_true,pred,zero_division=0),'recall':recall_score(y_true,pred,zero_division=0),'f1':f1_score(y_true,pred,zero_division=0),'roc_auc':roc_auc_score(y_true,prob) if y_true.nunique()>1 else np.nan})
    return result,pred,prob

performance=[]; outputs={}
for name,batch in batches.items():
    metrics,pred,prob=evaluate_batch(batch); metrics['batch']=name
    performance.append(metrics); outputs[name]={'predictions':pred,'probabilities':prob}
performance_df=pd.DataFrame(performance).set_index('batch')
display(performance_df.round(4))

## 7. Visualize feature and prediction distributions

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,col in zip(axes,['feature_1','feature_2','feature_4']):
    ax.hist(train[col].dropna(),bins=35,density=True,alpha=.45,label='Reference')
    for name,batch in batches.items(): ax.hist(batch[col].dropna(),bins=35,density=True,histtype='step',linewidth=1.5,label=name)
    ax.set_title(col); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

plt.figure(figsize=(8,4))
for name,out in outputs.items(): plt.hist(out['probabilities'],bins=30,alpha=.4,label=name)
plt.title('Predicted probability distributions'); plt.xlabel('Probability of class 1'); plt.ylabel('Count'); plt.legend(); plt.tight_layout(); plt.show()

## 8. Alert summary and export

Thresholds below are demonstration values. Tune them using historical backtests and the cost of false alarms.

In [ ]:
def make_alert(name,batch):
    alerts=[]
    flagged=numeric_reports[name].loc[numeric_reports[name].drift_detected,'feature'].tolist()
    if flagged: alerts.append('Numeric drift: '+', '.join(flagged))
    if cat_reports[name]['drift_detected']: alerts.append('Categorical drift: region')
    missing=batch[features].isna().mean()*100
    high=missing[missing>=2]
    if len(high): alerts.append('High missingness: '+', '.join(f'{k} ({v:.1f}%)' for k,v in high.items()))
    if 'f1' in performance_df.columns and pd.notna(performance_df.loc[name,'f1']):
        drop=baseline['f1']-performance_df.loc[name,'f1']
        if drop>=.08: alerts.append(f'F1 dropped by {drop:.3f} versus holdout baseline')
    return {'batch':name,'status':'ALERT' if alerts else 'OK','alert_count':len(alerts),'alerts':alerts or ['No configured thresholds triggered']}

alerts_df=pd.DataFrame([make_alert(name,batch) for name,batch in batches.items()])
summary=performance_df.reset_index().merge(alerts_df[['batch','status','alert_count']],on='batch',how='left')
display(alerts_df); display(summary.round(4))
summary.to_csv('monitoring_summary.csv',index=False)
summary.to_json('monitoring_summary.json',orient='records',indent=2)
print('Exported monitoring_summary.csv and monitoring_summary.json')

## Productionization checklist
- Define reference window, refresh policy, and data lineage.
- Validate schema, feature types, ranges, missingness, and duplicates.
- Consider PSI, Jensen–Shannon divergence, Wasserstein distance, and domain-specific tests.
- Account for multiple comparisons and alert fatigue.
- Separate data drift, prediction drift, concept drift, and performance degradation.
- Track delayed labels and model versions.
- Backtest thresholds before deploying alerts or retraining triggers.
- Add monitoring dashboards, scheduled jobs, secure storage, and notification integrations.

## Resume-ready summary
Built a Python ML monitoring workflow to track data quality, numeric and categorical drift, prediction behavior, and model performance across simulated production batches. Implemented KS and chi-square tests, alert summaries, visual diagnostics, and CSV/JSON reporting using pandas, SciPy, scikit-learn, and Matplotlib.